# TP 4 — Deep Learning et Transformers
### Partie IV de la conférence · Licence 1 → Master 2

> **TechForge Club · FST — Université de Nouakchott**  
> Conférence **« L’intelligence artificielle : au-delà de la théorie »** · الذكاء الاصطناعي : ما وراء النظرية  
> Animée par **Dr El Benany Med Mahmoud** · samedi 14 novembre 2026 · 12 h 30 · Amphi 5 – FST

**Trois parcours dans le même notebook — choisissez le vôtre :**

| Repère | Public conseillé | Ce que l’on vous demande |
|:---:|---|---|
| 🟢 **Découverte** | Licence 1 – Licence 2 | Exécuter, observer, modifier une valeur, répondre en une phrase. Aucune démonstration exigée. |
| 🟡 **Approfondissement** | Licence 3 | Refaire les calculs à la main, compléter du code, interpréter les résultats. |
| 🔴 **Expert** | Master 1 – Master 2 | Démontrer, implémenter sans bibliothèque, critiquer le protocole, mener un mini-projet. |

Chaque exercice **À faire** porte un repère. Un étudiant de licence peut s’arrêter aux 🟢 d’une section et passer à la suivante ; un étudiant de master traite les trois niveaux. Les cellules `verifier(...)` contrôlent votre réponse sans afficher la solution.

**Objectifs.** Calculer un neurone à la main, propager une entrée dans un réseau, mesurer une perte, rétropropager un gradient, **entraîner un réseau écrit entièrement en NumPy**, puis ouvrir la boîte des Transformers : embeddings, attention, masque causal, prédiction du token suivant et température.

**Diapositives associées :** Partie IV · 01 à 20.

**Prérequis.** 🟢 exécuter une cellule, multiplier et additionner ; 🟡 Python, vecteurs et matrices ; 🔴 dérivées partielles, règle de chaîne, NumPy.

**Aucune carte graphique ni bibliothèque de deep learning n’est nécessaire :** tout est écrit en NumPy pour que chaque calcul reste visible. Les valeurs d’embeddings et les poids non entraînés sont **aléatoires** et signalés comme tels.

| Section | Thème | Diapos | 🟢 | 🟡 | 🔴 | Durée |
|---|---|:---:|:---:|:---:|:---:|:---:|
| 1 | Neurone artificiel et activations | 01–03 | ✔ | ✔ | ✔ | 20 min |
| 2 | Perceptron multicouche et propagation avant | 04–05 | ✔ | ✔ | ✔ | 20 min |
| 3 | Fonctions de perte | 06 | ✔ | ✔ | ✔ | 15 min |
| 4 | Rétropropagation | 07 | ✔ | ✔ | ✔ | 25 min |
| 5 | Entraîner un réseau en NumPy | 08 | ✔ | ✔ | ✔ | 35 min |
| 6 | Calcul matriciel, matériel, kernel par blocs | 09–11 | ✔ | ✔ | ✔ | 15 min |
| 7 | Tokens, embeddings, positions | 12 | ✔ | ✔ | ✔ | 15 min |
| 8 | Attention et bloc Transformer | 13–15 | ✔ | ✔ | ✔ | 35 min |
| 9 | Modèle de langage, token suivant, température | 16–18 | ✔ | ✔ | ✔ | 30 min |
| 10 | Usage responsable, mini-projets, barème | 19–20 | ✔ | ✔ | ✔ | projet |

In [ ]:
# Installation (à décommenter une seule fois si nécessaire) :
# %pip install numpy pandas matplotlib scikit-learn ipywidgets
# --- Outils communs aux quatre TP de la conférence ---
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display
SEED = 42
rng = np.random.default_rng(SEED)
plt.rcParams.update({'figure.figsize': (7, 4), 'axes.grid': True, 'grid.alpha': .3})

def verifier(nom, reponse, attendu, tol=1e-2):
    """Compare votre réponse à la valeur attendue, sans l'afficher."""
    if reponse is None:
        print(f'⏳ {nom} : à compléter (remplacez None par votre réponse)'); return
    ok = np.allclose(np.asarray(reponse, dtype=float), np.asarray(attendu, dtype=float), atol=tol)
    print(f'✅ {nom} : correct' if ok else f'❌ {nom} : à revoir — reprenez le calcul')

def explorer(f, **curseurs):
    """Curseurs interactifs si ipywidgets est installé ; sinon exécution avec les valeurs par défaut.
    Chaque curseur est donné sous la forme nom=(min, max, pas, valeur_par_defaut)."""
    try:
        import ipywidgets as w
        ctl = {k: (w.FloatSlider if any(isinstance(x, float) for x in v) else w.IntSlider)(
                   min=v[0], max=v[1], step=v[2], value=v[3], description=k, continuous_update=False)
               for k, v in curseurs.items()}
        w.interact(f, **ctl)
    except ImportError:
        print('(ipywidgets absent : résultat pour les valeurs par défaut — changez-les dans l’appel pour explorer)')
        f(**{k: v[3] for k, v in curseurs.items()})
import time, math
from sklearn.datasets import make_moons, load_digits
from sklearn.model_selection import train_test_split
print('Environnement prêt. Graine =', SEED)

## 1 — Le neurone artificiel et les activations  *(diapos 01–03)*

Un neurone calcule une **combinaison affine** de ses entrées, puis applique une **activation** non linéaire :

$$z=\sum_j w_jx_j+b,\qquad a=\sigma(z)=\frac1{1+e^{-z}}.$$

**Exemple du cours.** $x=(2,1)$, $w=(0{,}5\,;-1)$, $b=0$ : $z=0{,}5\times2-1\times1=0$, donc $a=\sigma(0)=0{,}5$.

Trois activations courantes : $\mathrm{ReLU}(z)=\max(0,z)$, sigmoïde, $\tanh$. Pour $z=-2$ : ReLU $=0$, sigmoïde $\approx0{,}119$, $\tanh\approx-0{,}964$.

> Sans non-linéarité, une composition de couches affines reste une transformation affine.

In [ ]:
sigmoide = lambda z: 1 / (1 + np.exp(-z))
relu = lambda z: np.maximum(0, z)

def neurone(x, w, b):
    z = float(np.dot(w, x) + b)
    return z, float(sigmoide(z))

print('Exemple du cours : z, a =', neurone([2, 1], [0.5, -1], 0))
zs = np.linspace(-4, 4, 200)
for nom, f in [('ReLU', relu), ('Sigmoïde', sigmoide), ('tanh', np.tanh)]:
    plt.plot(zs, f(zs), label=nom)
plt.axvline(-2, ls=':', color='gray'); plt.ylim(-1.2, 3); plt.xlabel('z'); plt.legend(); plt.title('Trois fonctions d’activation'); plt.show()
print('z = −2 → ReLU =', relu(-2), '| sigmoïde =', round(float(sigmoide(-2)), 3), '| tanh =', round(float(np.tanh(-2)), 3))

In [ ]:
def voir_neurone(w1=0.5, w2=-1.0, b=0.0):
    z, a = neurone([2, 1], [w1, w2], b)
    print(f'x = (2, 1) | z = {w1}×2 + {w2}×1 + {b} = {z:.2f} | a = σ(z) = {a:.3f}')

explorer(voir_neurone, w1=(-2.0, 2.0, 0.5, 0.5), w2=(-2.0, 2.0, 0.5, -1.0), b=(-2.0, 2.0, 0.5, 0.0))

**À faire**

- 🟢 Avec l’explorateur, trouvez des poids qui donnent une sortie supérieure à 0,9. Que fait le biais $b$ ?
- 🟡 Calculez à la main $z$ et $a$ pour $x=(2,3)$, $w=(1,2)$, $b=-1$ (exemple de la partie I).
- 🔴 Montrez que deux couches affines sans activation, $W_2(W_1x+b_1)+b_2$, équivalent à une seule couche affine. Donnez sa matrice et son biais.

In [ ]:
mon_z, mon_a = None, None           # 🟡
verifier('z', mon_z, neurone([2, 3], [1, 2], -1)[0]); verifier('a = σ(z)', mon_a, neurone([2, 3], [1, 2], -1)[1], tol=1e-3)

## 2 — Perceptron multicouche et propagation avant  *(diapos 04–05)*

$$h=\mathrm{ReLU}(W_1x+b_1),\qquad \hat y=W_2h+b_2.$$

**Exemple du cours.** $x=\begin{pmatrix}2\\1\end{pmatrix}$, $W_1=\begin{pmatrix}1&-1\\0&2\end{pmatrix}$, $b_1=\begin{pmatrix}0\\-1\end{pmatrix}$ : $W_1x+b_1=\begin{pmatrix}1\\1\end{pmatrix}$, $h=\begin{pmatrix}1\\1\end{pmatrix}$. Avec $W_2=(2\;\;-1)$, $b_2=0$ : $\hat y=2\times1-1\times1=1$.

Les dimensions doivent s’emboîter : $W_1$ a une ligne par neurone caché et une colonne par entrée.

In [ ]:
W1 = np.array([[1., -1.], [0., 2.]]); b1 = np.array([0., -1.]); W2 = np.array([[2., -1.]]); b2 = np.array([0.])

def propagation_avant(x, detail=True):
    z1 = W1 @ x + b1; h = relu(z1); y = W2 @ h + b2
    if detail: print(f'x = {x} → W₁x + b₁ = {z1} → h = ReLU = {h} → ŷ = {y[0]}')
    return y[0]

assert propagation_avant(np.array([2., 1.])) == 1
propagation_avant(np.array([1., 3.])); propagation_avant(np.array([-1., 0.]))
print('Paramètres du réseau :', W1.size + b1.size + W2.size + b2.size)

**À faire**

- 🟢 Pour quelle entrée de la cellule précédente un neurone caché est-il « éteint » (valeur 0 après ReLU) ?
- 🟡 Calculez à la main la sortie pour $x=(3,2)$.
- 🔴 Combien de paramètres possède un réseau $64\to32\to10$ (entièrement connecté, avec biais) ? Écrivez la formule générale pour des couches de tailles $n_0,n_1,\dots,n_L$.

In [ ]:
ma_sortie = None                    # 🟡
verifier('ŷ pour x = (3, 2)', ma_sortie, propagation_avant(np.array([3., 2.]), detail=False))
mes_parametres = None               # 🔴
verifier('Paramètres 64→32→10', mes_parametres, 64 * 32 + 32 + 32 * 10 + 10, tol=.5)

## 3 — Fonctions de perte : mesurer l’erreur  *(diapo 06)*

**Régression :** $L_{MSE}=\dfrac1n\sum_i(y_i-\hat y_i)^2$.  **Classification binaire :** $L_{BCE}=-\dfrac1n\sum_i\big[y_i\ln p_i+(1-y_i)\ln(1-p_i)\big]$.

**Exemples du cours.** $y=(2,4)$, $\hat y=(3,2)$ : MSE $=\dfrac{1^2+(-2)^2}2=2{,}5$. Pour $y=1$ et $p=0{,}8$ : $L=-\ln0{,}8\approx0{,}223$.

> La perte guide l’optimisation ; les métriques d’évaluation jugent la qualité sur des données non vues.

In [ ]:
mse = lambda y, yh: float(np.mean((np.asarray(y) - np.asarray(yh)) ** 2))
bce = lambda y, p: float(-np.mean(np.asarray(y) * np.log(p) + (1 - np.asarray(y)) * np.log(1 - np.asarray(p))))
print('MSE =', mse([2, 4], [3, 2]), '| BCE(y = 1, p = 0,8) =', round(bce([1], [0.8]), 3))
ps = np.linspace(.01, .99, 200)
plt.plot(ps, -np.log(ps), label='y = 1 : −ln(p)'); plt.plot(ps, -np.log(1 - ps), label='y = 0 : −ln(1 − p)')
plt.xlabel('probabilité prédite p'); plt.ylabel('perte'); plt.legend(); plt.title('L’entropie croisée punit les erreurs confiantes'); plt.show()

**À faire**

- 🟢 La perte est-elle plus grande pour $p=0{,}9$ ou $p=0{,}1$ quand la vraie classe est 1 ? Pourquoi ?
- 🟡 Calculez à la main la BCE moyenne pour $y=(1,0)$ et $p=(0{,}9\,;0{,}2)$.
- 🔴 Montrez que la BCE est l’opposé de la log-vraisemblance d’un modèle de Bernoulli, et que sa dérivée par rapport au logit $z$ vaut $p-y$ lorsque $p=\sigma(z)$.

In [ ]:
ma_bce = None                       # 🟡
verifier('BCE moyenne', ma_bce, bce([1, 0], [0.9, 0.2]), tol=1e-3)

## 4 — Rétropropagation : la règle de chaîne  *(diapo 07)*

Exemple à un paramètre : $\hat y=wx$, $L=\tfrac12(\hat y-y)^2$.

$$\frac{\partial L}{\partial w}=\frac{\partial L}{\partial\hat y}\cdot\frac{\partial\hat y}{\partial w}=(\hat y-y)\,x,\qquad w\leftarrow w-\eta\,\frac{\partial L}{\partial w}.$$

**Exemple du cours.** $x=2$, $y=6$, $w=2$ : $\hat y=4$, $\partial L/\partial w=(4-6)\times2=-4$ ; avec $\eta=0{,}1$ : $w'=2-0{,}1\times(-4)=2{,}4$.

In [ ]:
def un_pas(w, x=2., y=6., eta=.1, detail=True):
    y_chap = w * x; perte = .5 * (y_chap - y) ** 2          # avant
    grad = (y_chap - y) * x                                  # arrière
    if detail: print(f'w = {w:.3f} | ŷ = {y_chap:.3f} | L = {perte:.4f} | ∂L/∂w = {grad:.3f} | nouveau w = {w - eta * grad:.3f}')
    return w - eta * grad

w = 2.
for _ in range(6): w = un_pas(w)

# Vérification numérique du gradient : (L(w + ε) − L(w − ε)) / 2ε
L = lambda w: .5 * (w * 2. - 6.) ** 2; eps = 1e-5
print('Gradient numérique en w = 2 :', round((L(2 + eps) - L(2 - eps)) / (2 * eps), 4), '| gradient analytique : −4')

**À faire**

- 🟢 Vers quelle valeur $w$ converge-t-il ? Pourquoi est-ce la bonne ?
- 🟡 Faites à la main deux pas à partir de $w=0$ avec $\eta=0{,}1$.
- 🔴 Ajoutez un biais : $\hat y=wx+b$. Calculez $\partial L/\partial b$, puis vérifiez numériquement. Que vaut $\eta$ maximal avant divergence pour ce problème à un paramètre ?

In [ ]:
mon_w2 = None                       # 🟡 valeur de w après deux pas depuis w = 0
verifier('w après deux pas', mon_w2, un_pas(un_pas(0., detail=False), detail=False), tol=1e-3)

## 5 — Entraîner un réseau de neurones écrit en NumPy  *(diapo 08)*

On classe deux « croissants » entrelacés — une frontière qu’aucune droite ne peut tracer — avec un réseau $2\to h\to1$ :

$$H=\mathrm{ReLU}(XW_1^\top+b_1),\qquad p=\sigma(HW_2^\top+b_2),\qquad L=\text{BCE}(y,p).$$

**Gradients (rétropropagation).** Avec $\delta=(p-y)/n$ : $\;\nabla_{W_2}=\delta^\top H$, $\;\nabla_{b_2}=\sum\delta$, $\;\Delta=(\delta W_2)\odot\mathbb 1[Z_1>0]$, $\;\nabla_{W_1}=\Delta^\top X$, $\;\nabla_{b_1}=\sum\Delta$.

On suit la perte sur l’**entraînement** et sur la **validation** : une perte d’entraînement qui baisse ne garantit pas la généralisation.

In [ ]:
def initialiser(d, h, graine=SEED):
    g = np.random.default_rng(graine)
    return {'W1': g.normal(0, np.sqrt(2 / d), (h, d)), 'b1': np.zeros(h), 'W2': g.normal(0, np.sqrt(1 / h), (1, h)), 'b2': np.zeros(1)}

def avant(P, X):
    Z1 = X @ P['W1'].T + P['b1']; H = relu(Z1); p = sigmoide(H @ P['W2'].T + P['b2'])[:, 0]
    return Z1, H, p

def perte_bce(p, y):
    p = np.clip(p, 1e-9, 1 - 1e-9); return float(-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)))

def gradients(P, X, y):
    Z1, H, p = avant(P, X); delta = ((p - y) / len(y))[:, None]
    D = (delta @ P['W2']) * (Z1 > 0)
    return {'W1': D.T @ X, 'b1': D.sum(0), 'W2': delta.T @ H, 'b2': delta.sum(0)}

def entrainer(X, y, Xv, yv, h=8, eta=.5, epoques=300, lot=32, graine=SEED):
    P = initialiser(X.shape[1], h, graine); g = np.random.default_rng(graine); hist = []
    for _ in range(epoques):
        ordre = g.permutation(len(y))
        for i in range(0, len(y), lot):                       # mini-lots
            idx = ordre[i:i + lot]; G = gradients(P, X[idx], y[idx])
            for k in P: P[k] -= eta * G[k]
        hist.append((perte_bce(avant(P, X)[2], y), perte_bce(avant(P, Xv)[2], yv)))
    return P, np.array(hist)

Xm, ym = make_moons(n_samples=400, noise=.22, random_state=SEED)
Xtr, Xva, ytr, yva = train_test_split(Xm, ym, test_size=.3, random_state=SEED)
P, hist = entrainer(Xtr, ytr, Xva, yva)
exactitude = lambda P, X, y: float(np.mean((avant(P, X)[2] >= .5) == y))
print(f'Exactitude — entraînement : {exactitude(P, Xtr, ytr):.3f} | validation : {exactitude(P, Xva, yva):.3f}')

In [ ]:
# Vérification du gradient par différences finies (indispensable quand on écrit sa propre rétropropagation)
P0 = initialiser(2, 4); G = gradients(P0, Xtr[:20], ytr[:20]); eps = 1e-6; ecart = 0
for k in P0:
    it = np.nditer(P0[k], flags=['multi_index'])
    for _ in it:
        i = it.multi_index; v = P0[k][i]
        P0[k][i] = v + eps; lp = perte_bce(avant(P0, Xtr[:20])[2], ytr[:20])
        P0[k][i] = v - eps; lm = perte_bce(avant(P0, Xtr[:20])[2], ytr[:20]); P0[k][i] = v
        ecart = max(ecart, abs((lp - lm) / (2 * eps) - G[k][i]))
print(f'Écart maximal entre gradient analytique et numérique : {ecart:.2e}'); assert ecart < 1e-5

In [ ]:
def frontiere(P, ax, titre):
    gx, gy = np.meshgrid(np.linspace(-1.8, 2.8, 200), np.linspace(-1.4, 1.9, 200))
    ax.contourf(gx, gy, avant(P, np.c_[gx.ravel(), gy.ravel()])[2].reshape(gx.shape), levels=20, cmap='RdBu', alpha=.6)
    ax.scatter(*Xtr.T, c=ytr, cmap='RdBu', s=12, edgecolor='k', linewidth=.3); ax.set_title(titre); ax.grid(False)

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].plot(hist[:, 0], label='entraînement'); ax[0].plot(hist[:, 1], label='validation'); ax[0].set_xlabel('époque'); ax[0].set_ylabel('perte'); ax[0].legend(); ax[0].set_title('Courbes d’apprentissage')
frontiere(P, ax[1], '8 neurones cachés'); frontiere(entrainer(Xtr, ytr, Xva, yva, h=1)[0], ax[2], '1 neurone caché : frontière presque droite')
plt.tight_layout(); plt.show()

In [ ]:
# Effet du taux d'apprentissage
for eta in [.01, .5, 3.0]:
    _, hh = entrainer(Xtr, ytr, Xva, yva, eta=eta, epoques=120)
    plt.plot(hh[:, 0], label=f'η = {eta}')
    print(f'η = {eta:<4} | perte finale = {hh[-1, 0]:.3f} | plus forte remontée d’une époque à la suivante = {np.diff(hh[:, 0]).max():.3f}')
plt.xlabel('époque'); plt.ylabel('perte d’entraînement'); plt.legend(); plt.title('η trop faible : lent · η trop élevé : remontées brutales'); plt.show()

**À faire**

- 🟢 Comparez les trois courbes : laquelle est lente ? laquelle remonte brutalement par moments ? Pourquoi une courbe instable est-elle risquée, même si elle finit bas ?
- 🟡 Entraînez avec `h = 2`, `8`, `64`. Comparez exactitude d’entraînement et de validation, puis regardez les frontières.
- 🟡 Passez de mini-lots de 32 à un seul lot (`lot = len(ytr)`). La courbe est-elle plus lisse ? plus rapide ?
- 🔴 Ajoutez une régularisation L2 ($\lambda\|W\|^2$) à la perte et aux gradients, puis un **arrêt précoce** fondé sur la perte de validation. Vérifiez vos gradients par différences finies.
- 🔴 Remplacez ReLU par $\tanh$ : quelle ligne de `gradients` faut-il changer ?

## 6 — Calcul matriciel, matériel et kernel par blocs  *(diapos 09–11)*

Une couche dense appliquée à un lot est un produit de matrices : $Y=XW+B$, avec $X\in\mathbb R^{m\times d}$, $W\in\mathbb R^{d\times h}$, $Y\in\mathbb R^{m\times h}$. Pour $m=32$, $d=128$, $h=64$ : environ $32\times128\times64=262\,144$ multiplications, hors biais et activation.

Ces opérations régulières se prêtent au parallélisme : **CPU** pour l’orchestration, **GPU** pour le débit, **NPU** pour l’inférence embarquée. Aucun n’est toujours supérieur : on mesure latence, débit, mémoire et énergie sur la charge réelle.

In [ ]:
m, d, h = 32, 128, 64
Xl, Wl = rng.normal(size=(m, d)), rng.normal(size=(d, h))
t = time.perf_counter()
Y_boucles = np.array([[sum(Xl[i, k] * Wl[k, j] for k in range(d)) for j in range(h)] for i in range(m)])
t_boucles = time.perf_counter() - t
t = time.perf_counter(); Y_vect = Xl @ Wl; t_vect = time.perf_counter() - t
assert np.allclose(Y_boucles, Y_vect)
print(f'{m * d * h:,} multiplications | boucles Python : {t_boucles * 1000:.0f} ms | produit matriciel : {t_vect * 1000:.3f} ms'.replace(',', ' '))

def kernel_addition(a, b, bloc=8):
    """Simulation d'un kernel : chaque bloc serait exécuté en parallèle sur GPU ; le masque protège la fin du vecteur."""
    c = np.zeros(len(a)); nb = math.ceil(len(a) / bloc)
    for i in range(nb):
        idx = i * bloc + np.arange(bloc); masque = idx < len(a); c[idx[masque]] = a[idx[masque]] + b[idx[masque]]
    return c, nb, nb * bloc - len(a)

c, nb, masquees = kernel_addition(np.arange(18.), np.ones(18))
print(f'N = 18, blocs de 8 → {nb} blocs, {masquees} positions masquées')

**À faire**

- 🟢 Combien de multiplications pour $m=64$, $d=256$, $h=128$ ?
- 🟡 Combien de blocs et de positions masquées pour $N=1000$ et des blocs de 128 ?
- 🔴 Un kernel accélère une opération ; il ne corrige pas un algorithme. Donnez un exemple où un calcul très rapide produit un modèle inutilisable.

In [ ]:
mes_blocs, mes_masquees = None, None    # 🟡
verifier('Nombre de blocs', mes_blocs, math.ceil(1000 / 128), tol=.1); verifier('Positions masquées', mes_masquees, math.ceil(1000 / 128) * 128 - 1000, tol=.1)

## 7 — Des séquences aux vecteurs : tokens, embeddings, positions  *(diapo 12)*

Le texte est découpé en **tokens** ; chacun reçoit un identifiant, puis un vecteur lu dans une table apprise $E\in\mathbb R^{V\times d}$ : $e_i=E[t_i]$. On ajoute une information de **position**, sinon « le chat dort » et « dort le chat » seraient indiscernables.

Ici la table $E$ est **aléatoire** (non entraînée) : elle montre la mécanique, pas du sens.

In [ ]:
phrase = 'le chat dort sur le tapis'.split()
vocabulaire = sorted(set(phrase)); ident = {m_: i for i, m_ in enumerate(vocabulaire)}
tokens = [ident[m_] for m_ in phrase]; d_modele = 8
E = np.random.default_rng(SEED).normal(0, 1, (len(vocabulaire), d_modele))            # table d'embeddings (aléatoire)

def positions(n, d):
    pos = np.arange(n)[:, None]; i = np.arange(d)[None, :]
    angle = pos / 10000 ** (2 * (i // 2) / d)
    return np.where(i % 2 == 0, np.sin(angle), np.cos(angle))

X_seq = E[tokens] + positions(len(tokens), d_modele)
print('Vocabulaire :', ident); print('Tokens :', tokens, '| forme de la séquence d’entrée :', X_seq.shape)
print('« le » en position 0 et en position 4 ont le même embedding :', np.allclose(E[tokens[0]], E[tokens[4]]), '| la même entrée après ajout de la position :', np.allclose(X_seq[0], X_seq[4]))

**À faire**

- 🟢 Combien de tokens et combien de mots **différents** dans la phrase ?
- 🟡 Combien de coefficients contient la table $E$ pour un vocabulaire de 50 000 tokens et $d=768$ ?
- 🔴 Tracez la matrice des encodages de position pour 50 positions et $d=64$ (`plt.imshow`). Pourquoi des sinusoïdes de fréquences différentes permettent-elles de distinguer les positions ?

In [ ]:
mes_coefficients = None             # 🟡
verifier('Taille de la table E', mes_coefficients, 50000 * 768, tol=.5)

## 8 — Attention et bloc Transformer  *(diapos 13–15)*

$$\mathrm{Attention}(Q,K,V)=\mathrm{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V.$$

Chaque token émet une **requête** $Q$, comparée aux **clés** $K$ de tous les tokens ; les poids obtenus servent à mélanger les **valeurs** $V$.

**Calcul manuel du cours.** Deux tokens, $d_k=2$, $Q=K=V=I_2$ : $\dfrac{QK^\top}{\sqrt2}=\begin{pmatrix}0{,}707&0\\0&0{,}707\end{pmatrix}$ ; $\mathrm{softmax}(0{,}707\,;0)\approx(0{,}670\,;0{,}330)$ ; donc $A\approx\begin{pmatrix}0{,}670&0{,}330\\0{,}330&0{,}670\end{pmatrix}$ et $AV=A$.

> Les poids d’attention sont des pondérations contextuelles ; ils ne constituent pas une explication causale directe des décisions du modèle.

In [ ]:
def softmax(z, axis=-1):
    z = z - z.max(axis=axis, keepdims=True); e = np.exp(z); return e / e.sum(axis=axis, keepdims=True)

def attention(Q, K, V, masque=None):
    scores = Q @ K.T / np.sqrt(K.shape[1])
    if masque is not None: scores = np.where(masque, scores, -1e9)       # positions interdites
    A = softmax(scores); return A @ V, A

sortie, A = attention(np.eye(2), np.eye(2), np.eye(2))
print('Poids d’attention :\n', A.round(3)); assert np.allclose(A, [[.670, .330], [.330, .670]], atol=1e-3)

In [ ]:
# Self-attention sur la phrase (poids aléatoires, non entraînés) : sans masque (encodeur) et avec masque causal (décodeur)
g = np.random.default_rng(1); Wq, Wk, Wv = (g.normal(0, .6, (d_modele, d_modele)) for _ in range(3))
n = len(phrase); causal = np.tril(np.ones((n, n), dtype=bool))
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for a_, masque, titre in [(ax[0], None, 'Encodeur : chaque token voit toute la séquence'), (ax[1], causal, 'Décodeur causal : pas de regard vers le futur')]:
    _, A = attention(X_seq @ Wq, X_seq @ Wk, X_seq @ Wv, masque)
    im = a_.imshow(A, cmap='viridis', vmin=0, vmax=1); a_.set_xticks(range(n)); a_.set_xticklabels(phrase, rotation=45); a_.set_yticks(range(n)); a_.set_yticklabels(phrase)
    a_.set_title(titre); a_.set_xlabel('token regardé'); a_.set_ylabel('token qui regarde'); a_.grid(False)
plt.colorbar(im, ax=ax, shrink=.8); plt.show()
print('Chaque ligne somme à 1 :', np.allclose(A.sum(1), 1), '| cases autorisées par le masque causal :', int(causal.sum()), 'sur', n * n)

### Un bloc Transformer complet, en vingt lignes

Attention (ici une seule tête), **connexion résiduelle**, **normalisation**, puis un petit réseau *feed-forward*, de nouveau avec connexion résiduelle et normalisation. Les blocs sont empilés $N$ fois. « Transformer » désigne une **famille** d’architectures : encodeur, décodeur causal, encodeur–décodeur.

In [ ]:
def normaliser(X, eps=1e-5):
    return (X - X.mean(-1, keepdims=True)) / (X.std(-1, keepdims=True) + eps)

def bloc_transformer(X, params, masque=None):
    Wq, Wk, Wv, Wo, F1, c1, F2, c2 = params
    att, A = attention(X @ Wq, X @ Wk, X @ Wv, masque)
    X = normaliser(X + att @ Wo)                               # résiduel + normalisation
    ff = relu(X @ F1 + c1) @ F2 + c2                           # réseau feed-forward, position par position
    return normaliser(X + ff), A

g = np.random.default_rng(2); dm, dff = d_modele, 4 * d_modele
params = [g.normal(0, .5, s) for s in [(dm, dm)] * 4 + [(dm, dff), (dff,), (dff, dm), (dm,)]]
S = X_seq
for couche in range(3):
    S, _ = bloc_transformer(S, params, causal)
print('Entrée :', X_seq.shape, '→ après 3 blocs :', S.shape, '| paramètres d’un bloc :', sum(p.size for p in params))

**À faire**

- 🟢 Dans la carte du décodeur causal, pourquoi la moitié supérieure droite est-elle vide ?
- 🟡 Calculez à la main $\mathrm{softmax}(1\,;0)$ puis $\mathrm{softmax}(2\,;0)$. Que se passe-t-il quand l’écart entre scores augmente ?
- 🟡 Sur 5 tokens, combien de cases le masque causal autorise-t-il ?
- 🔴 Implémentez l’attention **multi-têtes** : découper $d$ en $h$ têtes, appliquer `attention` à chacune, concaténer. Vérifiez les dimensions.
- 🔴 Pourquoi diviser par $\sqrt{d_k}$ ? Simulez des vecteurs aléatoires de dimension 4, 64, 1024 et observez l’entropie des poids d’attention avec et sans cette division.

In [ ]:
ma_softmax = None                   # 🟡 softmax(1 ; 0) sous forme de liste de deux valeurs
verifier('softmax(1 ; 0)', ma_softmax, softmax(np.array([1., 0.])), tol=1e-3)
mes_cases = None                    # 🟡
verifier('Cases autorisées sur 5 tokens', mes_cases, 15, tol=.1)

## 9 — Modèle de langage : prédire le token suivant  *(diapos 16–18)*

Objectif d’entraînement autorégressif :

$$L(\theta)=-\sum_{t}\log p_\theta(x_t\mid x_1,\dots,x_{t-1}).$$

Version la plus simple : un **modèle de bigrammes de mots** — on estime $p(\text{mot suivant}\mid\text{mot courant})$ en comptant. Un LLM remplace ce comptage par un Transformer et un contexte long, mais l’objectif est le même : il apprend ce qui est **probable**, pas ce qui est **vrai**.

In [ ]:
corpus = ("le ciel est bleu . le ciel est gris . la mer est bleue . le chat dort . le chat boit du lait . "
          "le modele predit le mot suivant . le ciel est clair . la machine calcule . le chat est gris .").split()
vocab = sorted(set(corpus)); iv = {m_: i for i, m_ in enumerate(vocab)}
comptes = np.full((len(vocab), len(vocab)), .01)                         # lissage
for m1, m2 in zip(corpus, corpus[1:]): comptes[iv[m1], iv[m2]] += 1
logits_table = np.log(comptes)

def distribution(mot, temperature=1.0):
    return softmax(logits_table[iv[mot]] / temperature)

p = distribution('est'); top = np.argsort(-p)[:5]
print('Après « est » :', {vocab[i]: round(float(p[i]), 3) for i in top})
print('Perte si le mot attendu est « bleu » : −ln p =', round(-math.log(p[iv['bleu']]), 3), '| si c’est « dort » :', round(-math.log(p[iv['dort']]), 3))

### Température

$$p_i(T)=\frac{\exp(z_i/T)}{\sum_j\exp(z_j/T)},\qquad T>0.$$

**Tableau du cours**, logits $z=(2,1,0)$ : $T=0{,}5\to(0{,}867\,;0{,}117\,;0{,}016)$ ; $T=1\to(0{,}665\,;0{,}245\,;0{,}090)$ ; $T=2\to(0{,}506\,;0{,}307\,;0{,}186)$.

> La température modifie la diversité des sorties, pas la qualité factuelle du modèle.

In [ ]:
z = np.array([2., 1., 0.])
display(pd.DataFrame({f'T = {T}': softmax(z / T).round(3) for T in [0.5, 1, 2]}, index=['token 1', 'token 2', 'token 3']).T)
assert np.allclose(softmax(z / .5), [.867, .117, .016], atol=1e-3) and np.allclose(softmax(z / 2), [.506, .307, .186], atol=1e-3)

def generer(debut='le', longueur=12, temperature=1.0, graine=SEED):
    g = np.random.default_rng(graine); mots = [debut]
    for _ in range(longueur): mots.append(vocab[g.choice(len(vocab), p=distribution(mots[-1], temperature))])
    return ' '.join(mots)

for T in [0.2, 1.0, 3.0]: print(f'T = {T} → {generer(temperature=T)}')

def voir_temperature(T=1.0):
    p = softmax(z / T); plt.bar(['token 1', 'token 2', 'token 3'], p); plt.ylim(0, 1); plt.title(f'T = {T:.1f} → p = {p.round(3)}'); plt.show()

explorer(voir_temperature, T=(0.1, 3.0, 0.1, 1.0))

**À faire**

- 🟢 Générez plusieurs phrases en changeant `graine`. Le modèle « sait »-il de quelle couleur est le ciel ? Que répète-t-il, en réalité ?
- 🟡 Calculez à la main les probabilités pour $z=(2,1,0)$ et $T=1$, puis vérifiez.
- 🟡 Ajoutez la phrase « le ciel est vert . » dix fois dans le corpus. Que prédit alors le modèle après « est » ? Qu’en conclure sur la fiabilité d’un modèle entraîné sur des données fausses ?
- 🔴 Calculez la **perplexité** $\exp\big(-\frac1T\sum\log p\big)$ du modèle sur le corpus d’entraînement, puis sur une phrase nouvelle. Comment évolue-t-elle si l’on retire le lissage ?
- 🔴 Remplacez le tirage aléatoire par un **décodage glouton** (argmax), puis par un échantillonnage *top-k*. Comparez la diversité des sorties.

In [ ]:
mes_probas = None                   # 🟡 liste de trois valeurs pour z = (2, 1, 0), T = 1
verifier('softmax(2, 1, 0)', mes_probas, softmax(z), tol=1e-3)

## 10 — Évaluer et utiliser l’IA de manière responsable  *(diapos 19–20)*

| Évaluation technique | Risques et gouvernance |
|---|---|
| Données de test indépendantes | Hallucinations et biais |
| Exactitude, robustesse, calibration | Confidentialité et droits d’auteur |
| Latence, mémoire, coût énergétique | Sécurité des usages |
| Reproductibilité des expériences | Supervision humaine et traçabilité |

**Question de la conférence.** Un modèle donne une réponse convaincante mais sans source vérifiable : peut-on l’utiliser directement dans une décision importante ?

**À faire**

- 🟢 Citez deux situations où une réponse générée doit absolument être vérifiée avant usage.
- 🟡 Reprenez le réseau de la section 5 : sur quelles données faudrait-il l’évaluer avant de s’en servir ? Quelle métrique choisir si une erreur coûte plus cher que l’autre ?
- 🔴 Rédigez, en dix lignes, une fiche d’évaluation pour un assistant conversationnel destiné aux étudiants : données de test, métriques, risques, supervision humaine.

## Défis et mini-projets (à rendre)

| Niveau | Sujet | Travail demandé |
|---|---|---|
| 🟢 L1–L2 | **Défi découverte — mon premier neurone** | Choisir 3 entrées, des poids et un biais ; calculer $z$ et $a$ à la main et avec `neurone` ; tracer $a$ en fonction d’un poids ; commenter en 5 lignes. |
| 🟡 L3 | **Défi — la bonne taille de réseau** | Entraîner le réseau NumPy pour $h=1,2,4,8,16,64$ ; tracer exactitude d’entraînement et de validation ; identifier sous-apprentissage et surapprentissage. |
| 🔴 M1–M2 | **Mini-projet A — reconnaître des chiffres** | Étendre le réseau NumPy à 10 classes (softmax + entropie croisée) sur `load_digits()` ; découpage entraînement / validation / test ; arrêt précoce ; matrice de confusion ; comparaison avec `sklearn.neural_network.MLPClassifier`. |
| 🔴 M1–M2 | **Mini-projet B — mini-modèle de langage** | Modèle de $n$-grammes de caractères ou petit Transformer à un bloc sur un texte de votre choix ; perplexité sur un texte non vu ; effet de la température ; analyse des erreurs et des limites. |

In [ ]:
# MINI-PROJET A — cellule de départ (compléter les TODO)
chiffres = load_digits(); X_ch, y_ch = chiffres.data / 16.0, chiffres.target
print('Images :', X_ch.shape, '| classes :', np.unique(y_ch))
# TODO 1 : découpage entraînement / validation / test (stratifié)
# TODO 2 : sortie softmax à 10 classes, perte d'entropie croisée, gradients (à vérifier numériquement)
# TODO 3 : entraînement par mini-lots, courbes de perte, arrêt précoce
# TODO 4 : exactitude et matrice de confusion sur le test (une seule fois)
# TODO 5 : comparaison avec MLPClassifier et discussion des erreurs

## Barème (100 points)

| Critère | Points |
|---|---:|
| Calculs à la main justifiés (neurone, propagation, gradient, softmax) | 25 |
| Code correct, lisible, gradients vérifiés | 25 |
| Protocole entraînement / validation / test | 15 |
| Figures (axes, légendes) et interprétation | 15 |
| Esprit critique : limites, biais, usage responsable | 20 |

Pour les parcours 🟢 et 🟡, le barème s’applique aux exercices du niveau choisi et au défi correspondant.

## Autoévaluation

1. Que calcule un neurone ? À quoi sert l’activation ?
2. Pourquoi les dimensions des matrices doivent-elles être compatibles ?
3. Quelle différence entre une perte et une métrique d’évaluation ?
4. Énoncez la règle de chaîne sur l’exemple $\hat y=wx$.
5. Que se passe-t-il si le taux d’apprentissage est trop grand ? trop petit ?
6. Pourquoi suivre la perte de validation en plus de la perte d’entraînement ?
7. Qu’est-ce qu’un embedding ? Pourquoi ajouter la position ?
8. Que représentent $Q$, $K$ et $V$ dans l’attention ? Que fait le masque causal ?
9. Quel est l’objectif d’entraînement d’un LLM ? Garantit-il la véracité ?
10. Que change la température ? Que ne change-t-elle pas ?

## Conclusion de la conférence : les quatre questions du scientifique

1. Quelles **données** et quelles hypothèses ?
2. Quel **modèle** mathématique et quel objectif ?
3. Quel **calcul**, quel matériel, quelles ressources ?
4. Comment **vérifier** les résultats et les limites ?

Maîtriser l’IA, c’est comprendre, expérimenter, mesurer et exercer son jugement critique — pas seulement utiliser un outil.